# NB02 — fPINN de identificación con los datos del kit TCLab

**Proyecto:** fPINN + sintonía diferenciable de FOPID en el TCLab — *Control Engineering Practice*.
Versión **solo kit** (sin planta virtual), análoga a `NB01_identificacion_TCLab_kit.ipynb`. La versión con planta
virtual y barrido de ruido/duración sigue en `NB02_fPINN_simulacion.ipynb`.

> **No hace falta conectar el kit.** Este notebook solo lee los CSV que guardó NB01. Puede correrse en cualquier
> computador con Python + PyTorch (CPU basta).

**Qué hace.** Entrena la fPINN 2×2 (red de trayectorias + residuo de Grünwald–Letnikov, Adam → L-BFGS) con los
experimentos reales de NB01 y la compara con:

| Modelo | Para qué |
|---|---|
| **fPINN** (α y β aprendibles, pueden converger a 1) | sustituto diferenciable del método propuesto (P, B2, A1–A2 en NB03) |
| **PINN entera** (α = β = 1 fijos) | baseline **B3** |
| Físico entero / fraccionario por OE-LS (como NB01) | referencia directa con el mismo modelo |
| FOPDT, SOPDT, FFOPDT sobre G11 | SOPDT → B4, FFOPDT → B1 |

**Datos.** Busca `NB01_out/` (en esta carpeta o en `..`) y, si no está, el `NB01_resultados_<fecha>.zip` más reciente
y lo descomprime. Se usan `expA.csv` (escalones Q1), `expB.csv` (escalones Q2) y `expC.csv` (APRBS Q1).

**Particiones** (entrenamiento → validación):
- `AB-C` (principal): entrena con A y B, valida con C (APRBS, nunca vista).
- `BC-A`: entrena con B y C, valida con A. El análisis de NB01 mostró que la APRBS generaliza mejor; B se mantiene
  en el entrenamiento porque es el único experimento que excita Q2 (sin él α2 no es identificable).

**Ambiente variable.** En NB01 el error dominante no fue el orden del modelo sino la deriva de la temperatura de la
sala (1–2,5 °C, común a T1 y T2). Con `AMB_MODELS` se entrena:
- `'const'` (**principal**): T∞ constante, como NB01.
- `'drift'` (exploratorio): T∞(t) = T∞(0) + d(t), con d(t) lineal a tramos (nudos cada 600 s, d(0) = 0) aprendido junto
  con la red, regularizado en tamaño y suavidad. Solo afecta al entrenamiento. **Cuidado:** en la prueba intermedia
  (A+B completos, 6 nov) la deriva aprendida siguió a la entrada (+1,3 °C en A con Q1 alto, −1,9 °C en B con Q2 alto):
  absorbe error de modelo, no solo la sala. Bajó el RMSE de entrenamiento (0,91 → 0,55 °C) pero no el de validación.
  Por eso no es la configuración principal; sirve como ablación.

La **validación siempre simula el modelo identificado con ambiente constante**, porque la deriva del experimento de
validación no se conoce (igual que en lazo cerrado en NB03). Por eso el RMSE de validación incluye la deriva de ese
día y no baja de ~0,5–1 °C; lo que importa es la comparación entre modelos con los mismos datos.

**Entrenamiento final para NB03 (`FINAL_ABC = True`).** Aprobado por Luis (9 nov): el sustituto que usa NB03 es la
fPINN reentrenada con los **tres** experimentos A+B+C, ambiente constante, 3 semillas, sin validación externa (la
validación ya la dieron las particiones AB-C y BC-A). Guarda `NB02_out/kit/surrogates/ABC_const_s{0,1,2}.json` con
ϑ̂ de la fPINN, la PINN entera (B3), el físico entero y fraccionario OE-LS y los SISO (SOPDT → B4, FFOPDT → B1).
Son 6 entrenamientos (≈ 7–8 min cada uno en el portátil, ≈ 45–60 min en total).

**Cómo usarlo.** `MODE = 'quick'` (prueba, ~1–2 min). `MODE = 'full'`: 2 particiones × 2 modelos de ambiente ×
3 semillas × (fPINN + PINN entera) = 24 entrenamientos de ~3–5 min en una CPU de 4 núcleos (medido: 1500 Adam +
300 L-BFGS con A+B completos = 68 s), ≈ 1,5–2 h en total, más los ajustes OE-LS y SISO (~5–10 min).
**Empieza por la configuración principal** (`AB-C`, `const`), guarda cada corrida
en cuanto termina y **retoma** si se interrumpe. Al final crea `NB02_kit_resultados_<fecha>.zip` con todo `NB02_out/kit/`.
Requiere `%pip install torch`.

In [ ]:
import os
# Windows/Anaconda: torch y numpy/matplotlib traen cada uno su copia de OpenMP (libiomp5md.dll). Al cargarse las dos,
# el kernel muere sin mensaje ("The kernel appears to have died"). Esta línea lo evita; debe ir ANTES de importar
# numpy y torch. Si el kernel ya murió, reiniciarlo y correr desde aquí.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import json, time, math, glob, zipfile, datetime, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
import torch, torch.nn as nn
torch.set_default_dtype(torch.float64)

MODE = 'quick'                 # 'quick' (prueba) | 'full' (corrida completa, varias horas)
DATA_DIR = None                # None = busca NB01_out/ o NB01_resultados_*.zip; o una carpeta / un .zip concreto
OUT_DIR = os.path.join('NB02_out', 'kit')
os.makedirs(os.path.join(OUT_DIR, 'surrogates'), exist_ok=True)
N_THREADS = None               # p. ej. 4 para limitar los hilos de PyTorch; None = valor por defecto
if N_THREADS: torch.set_num_threads(N_THREADS)

SPLITS = {'AB-C': (['A', 'B'], ['C']),      # principal
          'BC-A': (['B', 'C'], ['A'])}
MAIN_SPLIT = 'AB-C'
FINAL_ABC = False              # True = entrenamiento FINAL para NB03: A+B+C juntos, ambiente constante, sin validación
                               # externa -> surrogates/ABC_const_s{0,1,2}.json (fPINN, PINN entera, OE-LS, SISO)
AMB_MODELS = ['const', 'drift']    # el primero es el principal
SEEDS = [0, 1, 2]
RUN_INT_PINN = True            # PINN entera (B3) en todas las configuraciones
D_TRAIN_MIN = None             # None = experimentos completos; p. ej. 30 = primeros 30 min de cada uno

# fPINN: mismos ajustes que en simulación (allí 4000 + 1500 recupera beta*; con menos no converge)
PINN = dict(n_adam=4000, n_lbfgs=1500, lr=2e-3, w_r=1.0, w_0=10.0, order_max=1.2, width=64, depth=4, n_ff=8, use_z=True)
AMB = dict(knot_s=600.0, w_mag=1e-1, w_smooth=1.0)   # deriva del ambiente (solo AMB_MODELS 'drift')
OE_NFEV, SISO_STARTS = 200, 3
L_SUR = None                   # memoria del sustituto (None = completa; ver NB02 simulación)
ORDER_TOL, GAIN_TOL = 0.05, 10.0   # criterio de H1 (el mismo de NB01)

if FINAL_ABC:
    SPLITS = {'ABC': (['A', 'B', 'C'], [])}; MAIN_SPLIT = 'ABC'; AMB_MODELS = ['const']
RES_CSV = os.path.join(OUT_DIR, 'resultados_kit.csv')
if MODE == 'quick':
    SPLITS = {MAIN_SPLIT: SPLITS[MAIN_SPLIT]}; SEEDS = [0]
    D_TRAIN_MIN = 15
    PINN.update(n_adam=400, n_lbfgs=30)
    OE_NFEV, SISO_STARTS = 15, 1
    RES_CSV = os.path.join(OUT_DIR, 'resultados_kit_quick.csv')
print('MODE =', MODE, '| particiones', list(SPLITS), '| ambiente', AMB_MODELS, '| semillas', SEEDS)

## 1. Carga de los datos de NB01

Columnas de NB01: `time, Q1, Q2, T1, T2, t_real` (Q en %, T en °C, una muestra por segundo). La rejilla se toma como
k·1 s (el paso real fue 0,96–1,04 s); se avisa si hay huecos. También se leen `metadatos_kit.json` y el ajuste físico
entero de NB01 (`parametros_fisicos.csv`), que se usa como punto de arranque de ϑ.

In [ ]:
TS = 1.0
REQ = ['expA.csv', 'expB.csv', 'expC.csv']

def find_data(data_dir=None):
    # 1) carpeta indicada o NB01_out/ ; 2) .zip indicado o el NB01_resultados_*.zip más reciente
    if data_dir and os.path.isdir(data_dir):
        cands = [data_dir]
    else:
        cands = ['NB01_out', os.path.join('..', 'NB01_out')]
    for d in cands:
        if all(os.path.exists(os.path.join(d, f)) for f in REQ): return d, d
    if data_dir and data_dir.endswith('.zip'):
        zips = [data_dir]
    else:
        zips = sorted(glob.glob('NB01_resultados_*.zip') + glob.glob(os.path.join('..', 'NB01_resultados_*.zip')),
                      key=os.path.getmtime)
    if zips:
        z = zips[-1]; dest = os.path.join('NB02_out', 'NB01_desde_zip')
        with zipfile.ZipFile(z) as f: f.extractall(dest)
        for root, _, files in os.walk(dest):
            if all(r in files for r in REQ): return root, z
    raise FileNotFoundError('No se encontró NB01_out/ ni NB01_resultados_*.zip con expA/expB/expC.csv. '
                            'Copiarlos junto a este notebook o poner DATA_DIR.')

def load_exp(path, name):
    df = pd.read_csv(path)
    miss = {'Q1', 'Q2', 'T1', 'T2'} - set(df.columns)
    if miss: raise ValueError(f'{path}: faltan columnas {miss}')
    df = df.dropna(subset=['Q1', 'Q2', 'T1', 'T2']).reset_index(drop=True)
    dt = np.diff(df['time'].values) if 'time' in df else np.ones(len(df) - 1)
    if dt.max() > 1.5 or dt.min() < 0.5:
        print(f'Aviso: exp. {name} tiene pasos de {dt.min():.2f}–{dt.max():.2f} s; se asume rejilla de 1 s')
    n = len(df)
    return dict(name=name, t=np.arange(n) * TS, Q1=df.Q1.values.astype(float), Q2=df.Q2.values.astype(float),
                T1=df.T1.values.astype(float), T2=df.T2.values.astype(float), dt_min=float(dt.min()), dt_max=float(dt.max()))

SRC_DIR, SRC = find_data(DATA_DIR)
EXPS = {nm: load_exp(os.path.join(SRC_DIR, f'exp{nm}.csv'), nm) for nm in 'ABC'}
META = json.load(open(os.path.join(SRC_DIR, 'metadatos_kit.json'))) if os.path.exists(os.path.join(SRC_DIR, 'metadatos_kit.json')) else {}
print('Datos de NB01 tomados de:', SRC)
for nm, e in EXPS.items():
    print(f'  exp. {nm}: {len(e["t"])} muestras, Q1 máx {e["Q1"].max():.0f} %, Q2 máx {e["Q2"].max():.0f} %, '
          f'T1 {e["T1"].min():.1f}–{e["T1"].max():.1f} °C, T2 {e["T2"].min():.1f}–{e["T2"].max():.1f} °C')
print('T_AMB (NB01):', META.get('T_AMB'), '| kit:', META.get('firmware'))

fig, ax = plt.subplots(2, 3, figsize=(14, 5), sharex='col')
for j, nm in enumerate('ABC'):
    e = EXPS[nm]
    ax[0, j].plot(e['t'], e['T1'], label='T1'); ax[0, j].plot(e['t'], e['T2'], label='T2')
    ax[0, j].set_title(f'Exp. {nm}'); ax[0, j].legend()
    ax[1, j].step(e['t'], e['Q1'], label='Q1'); ax[1, j].step(e['t'], e['Q2'], label='Q2'); ax[1, j].set_xlabel('t [s]')
ax[0, 0].set_ylabel('°C'); ax[1, 0].set_ylabel('%'); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig01_datos_kit.png'), dpi=150); plt.show(); plt.close('all')

## 2. Modelo 2×2 (el mismo de NB01, con órdenes α, β y ambiente T∞(t))

En variables de desviación x = T − T∞(0): m·cp·t_c^(α−1)·D^α x_H,i = F_i(x_H, Q_i, d(t)) y τs^β·D^β x_S,i = x_H,i − x_S,i,
con F_i = convección y radiación al ambiente T∞(0) + d(t) (área A, coeficiente U) + intercambio entre calentadores
(área As, coeficiente **Us**, como en NB01) + α_i·Q_i. Constantes geométricas iguales a NB01 (A = 12 cm², As = 2 cm²).
ϑ = (U, Us, α1, α2, τs, α, β). El sustituto para NB03 se simula con GL explícito, paso 1 s, memoria `L_SUR`.

In [ ]:
PHYS = dict(m=4e-3, cp=500.0, A=12e-4, As=2e-4, eps=0.9, sigma=5.67e-8, tc=1.0)   # = NB01 (kit)
TH_NAMES = ['U', 'Us', 'alpha1', 'alpha2', 'tau_s', 'alpha', 'beta']
POS_NAMES = TH_NAMES[:5]

# Arranque: ajuste físico entero de NB01 si existe (órdenes en 0,95 para no empezar en la cota)
TH0 = dict(U=3.0, Us=20.0, alpha1=0.011, alpha2=0.0055, tau_s=40.0, alpha=0.95, beta=0.95)
_pf = os.path.join(SRC_DIR, 'parametros_fisicos.csv')
if os.path.exists(_pf):
    _r = pd.read_csv(_pf); _r = _r[_r.modelo.str.contains('entero', case=False)]
    if len(_r):
        TH0.update({k: float(_r.iloc[0][k]) for k in POS_NAMES if k in _r and np.isfinite(_r.iloc[0][k])})
print('ϑ0 =', {k: round(v, 5) for k, v in TH0.items()})

def gl_w(a, n):
    return np.cumprod(np.r_[1.0, 1.0 - (a + 1.0) / np.arange(1, n)])

def F_heat(x1, x2, q1, q2, U, Us, a1, a2, TaK, d=0.0):
    # vale para numpy y torch; d = desviación del ambiente respecto a T∞(0)
    es, A, As = PHYS['eps'] * PHYS['sigma'], PHYS['A'], PHYS['As']
    T1, T2, Tam = TaK + x1, TaK + x2, TaK + d
    ex = Us * As * (T2 - T1) + es * As * (T2**4 - T1**4)
    return (U * A * (Tam - T1) + es * A * (Tam**4 - T1**4) + ex + a1 * q1,
            U * A * (Tam - T2) + es * A * (Tam**4 - T2**4) - ex + a2 * q2)

def sim_frac(Q1, Q2, th, Ta, d=None, L=None):
    # GL explícito del modelo 2x2, paso 1 s, entradas ZOH. d = deriva del ambiente por muestra (None = 0).
    U, Us, a1, a2, ts_, al, be = (th[k] for k in TH_NAMES)
    mcp = PHYS['m'] * PHYS['cp']; TaK = Ta + 273.15
    N = len(Q1); Lm = N if L is None else min(int(L), N); d = np.zeros(N) if d is None else d
    wa, wb = gl_w(al, Lm + 1), gl_w(be, Lm + 1)
    ca = TS**al / (mcp * PHYS['tc']**(al - 1)); cb = (TS / ts_)**be
    X = np.zeros((N + 1, 4))                  # xH1, xH2, xS1, xS2
    for k in range(1, N + 1):
        j = k - 1; xp = X[k - 1]
        f1, f2 = F_heat(xp[0], xp[1], Q1[j], Q2[j], U, Us, a1, a2, TaK, d[j])
        i0 = max(0, k - Lm)
        hist = X[k - 1:(i0 - 1 if i0 > 0 else None):-1]; n = hist.shape[0]
        ma = wa[1:n + 1] @ hist[:, :2]; mb = wb[1:n + 1] @ hist[:, 2:]
        X[k] = [ca * f1 - ma[0], ca * f2 - ma[1], cb * (xp[0] - xp[2]) - mb[0], cb * (xp[1] - xp[3]) - mb[1]]
    Xs = X[:N]
    return Ta + Xs[:, 2], Ta + Xs[:, 3], Xs

def amb0(e):
    # T∞(0): media de las 10 primeras muestras de ambos sensores (kit en equilibrio al arrancar)
    return float(np.mean(np.r_[e['T1'][:10], e['T2'][:10]]))

def cut(e, D_min):
    if D_min is None: return e
    n = int(D_min * 60 / TS); return {k: (v[:n] if isinstance(v, np.ndarray) else v) for k, v in e.items()}

# chequeo rápido: el ajuste de NB01 simulado sobre C (ambiente constante)
_e = EXPS['C']; _T1, _T2, _ = sim_frac(_e['Q1'], _e['Q2'], dict(TH0, alpha=1.0, beta=1.0), amb0(_e))
print('RMSE C con ϑ0 entero de NB01: T1 %.3f, T2 %.3f °C' % (np.sqrt(np.mean((_T1 - _e['T1'])**2)), np.sqrt(np.mean((_T2 - _e['T2'])**2))))

## 3. fPINN (y PINN entera)

Igual que en NB02 simulación: una red de trayectorias por experimento con t, 8 pares de Fourier y Q filtrada
(τ = 5, 20, 80, 300 s) → (x̂_H1, x̂_H2); sensor resuelto exacto (GL implícito por inversa de serie FFT, diferenciable en
τs y β); residuo del calentador con D^α (GL de memoria completa por FFT). Órdenes = 1,2·σ(·), así pueden llegar a 1
sin quedar en la cota. `learn_orders=False` fija α = β = 1 (**PINN entera, B3**).

Novedad para el kit (exploratoria): con `amb='drift'` cada experimento tiene su d(t) (lineal a tramos, nudos cada `AMB['knot_s']` s,
d(0) = 0), que entra en el residuo del calentador. Pérdida
𝓛 = 𝓛_data + w_r 𝓛_res + w_0 𝓛_0 + w_mag·media(d²) + w_smooth·media(Δ²d).

In [ ]:
def gl_w_t(a, n):
    k = torch.arange(1, n, dtype=torch.float64)
    return torch.cumprod(torch.cat([torch.ones(1), 1.0 - (a + 1.0) / k]), 0)

def fft_conv(a, x, N):
    n = 1 << (2 * N - 1).bit_length()
    return torch.fft.irfft(torch.fft.rfft(a, n).unsqueeze(1) * torch.fft.rfft(x, n, dim=0), n, dim=0)[:N]

def gl_deriv(x, a, h=TS):
    return fft_conv(gl_w_t(a, x.shape[0]), x, x.shape[0]) * h ** (-a)

def series_inv(a, N):
    b = 1.0 / a[:1]; m = 1
    while m < N:
        m = min(2 * m, N)
        bp = torch.cat([b, torch.zeros(m - b.shape[0])])
        corr = -fft_conv(a[:m], bp.unsqueeze(1), m)[:, 0]; corr[0] = corr[0] + 2.0
        b = fft_conv(bp, corr.unsqueeze(1), m)[:, 0]
    return b

def sensor_solve(xH, ts_, be, h=TS):
    N = xH.shape[0]; a = (ts_ / h) ** be * gl_w_t(be, N); a = torch.cat([a[:1] + 1.0, a[1:]])
    return fft_conv(series_inv(a, N), xH, N)

FILTER_TAUS = (5.0, 20.0, 80.0, 300.0)

def input_features(Q1, Q2):
    out = []
    for q in (Q1, Q2):
        for tau in FILTER_TAUS:
            a = math.exp(-TS / tau); z = np.zeros(len(q))
            for k in range(1, len(q)): z[k] = a * z[k - 1] + (1 - a) * q[k - 1]
            out.append(z / 100.0)
    return torch.tensor(np.stack(out, 1))

class TrajNet(nn.Module):
    def __init__(self, t_end, width, depth, n_ff, n_z, n_out, seed, scale=40.0):
        super().__init__()
        g = torch.Generator().manual_seed(seed); self.t_end, self.scale = t_end, scale
        self.register_buffer('freq', math.pi * torch.arange(1, n_ff + 1, dtype=torch.float64))
        layers, d = [], 1 + 2 * n_ff + n_z
        for _ in range(depth): layers += [nn.Linear(d, width), nn.Tanh()]; d = width
        self.net = nn.Sequential(*layers, nn.Linear(d, n_out))
        for m in self.net:
            if isinstance(m, nn.Linear): nn.init.xavier_normal_(m.weight, generator=g); nn.init.zeros_(m.bias)
    def forward(self, t, z=None):
        s = (t / self.t_end).unsqueeze(1)
        f = [s, torch.sin(s * self.freq), torch.cos(s * self.freq)] + ([] if z is None else [z])
        return self.scale * self.net(torch.cat(f, 1))

class Params(nn.Module):
    # theta = (U, Us, alpha1, alpha2, tau_s, alpha, beta): positivos por logaritmo, órdenes = order_max * sigmoid
    def __init__(self, th0, learn_orders=True, order_max=1.2):
        super().__init__()
        self.order_max, self.learn_orders = order_max, learn_orders
        self.logp = nn.Parameter(torch.log(torch.tensor([th0[k] for k in POS_NAMES])))
        logit = lambda o: math.log(o / order_max / (1 - o / order_max))
        if learn_orders: self.o = nn.Parameter(torch.tensor([logit(th0['alpha']), logit(th0['beta'])]))
    def get(self):
        U, Us, a1, a2, ts_ = torch.exp(self.logp)
        if self.learn_orders: al, be = self.order_max * torch.sigmoid(self.o)
        else: al = be = torch.tensor(1.0)
        return U, Us, a1, a2, ts_, al, be
    def as_dict(self):
        return dict(zip(TH_NAMES, [float(v.detach()) for v in self.get()]))

class Drift(nn.Module):
    # d(t) lineal a tramos con d(0) = 0 [°C]
    def __init__(self, t_end, knot_s):
        super().__init__()
        self.knot_s = knot_s; nk = int(math.ceil(t_end / knot_s)) + 1
        self.v = nn.Parameter(torch.zeros(nk - 1))
    def values(self):
        return torch.cat([torch.zeros(1), self.v])
    def forward(self, t):
        vals = self.values(); s = t / self.knot_s
        i = torch.clamp(torch.floor(s).long(), max=vals.shape[0] - 2); w = s - i
        return (1 - w) * vals[i] + w * vals[i + 1]
    def penalty(self):
        vals = self.values()
        sm = torch.mean(torch.diff(vals, n=2) ** 2) if vals.shape[0] > 2 else torch.tensor(0.0)
        return torch.mean(vals ** 2), sm

def heater_residual(X, q1, q2, P, TaK, d=None, t_ref=100.0):
    # residuo de m cp t_c^(alpha-1) D^alpha xH = F (entrada y ambiente del intervalo anterior), escalado a K
    U, Us, a1, a2, ts_, al, be = P.get()
    DH = gl_deriv(X[:, :2], al)
    sh = lambda v: torch.cat([v[:1] * 0, v[:-1]])
    dp = 0.0 if d is None else sh(d)
    f1, f2 = F_heat(X[:, 0], X[:, 1], sh(q1), sh(q2), U, Us, a1, a2, TaK, dp)
    c = PHYS['m'] * PHYS['cp'] * PHYS['tc'] ** (al - 1)
    return t_ref * torch.stack([DH[:, 0] - f1 / c, DH[:, 1] - f2 / c], 1)[1:]

def train_pinn(exps, th0=TH0, learn_orders=True, amb='drift', seed=0, n_adam=4000, n_lbfgs=1500, lr=2e-3, w_r=1.0,
               w_0=10.0, order_max=1.2, width=64, depth=4, n_ff=8, use_z=True, verbose=False):
    # Entrena la fPINN (o la PINN entera con learn_orders=False). Devuelve theta_hat, d(t) por experimento, historial, tiempo.
    torch.manual_seed(seed)
    P = Params(th0, learn_orders, order_max); data = []
    for i, e in enumerate(exps):
        Ta = amb0(e)
        data.append(dict(name=e['name'], t=torch.tensor(e['t']), q1=torch.tensor(e['Q1']), q2=torch.tensor(e['Q2']),
                         TaK=Ta + 273.15, y=torch.tensor(np.c_[e['T1'], e['T2']]) - Ta,
                         z=input_features(e['Q1'], e['Q2']) if use_z else None,
                         net=TrajNet(float(e['t'][-1]) + 1.0, width, depth, n_ff, 2 * len(FILTER_TAUS) if use_z else 0,
                                     2, seed=1000 * seed + i),
                         drift=Drift(float(e['t'][-1]) + 1.0, AMB['knot_s']) if amb == 'drift' else None))
    nets = nn.ModuleList([d['net'] for d in data])
    drifts = nn.ModuleList([d['drift'] for d in data if d['drift'] is not None])

    def states(d):
        X = d['net'](d['t'], d['z']); U, Us, a1, a2, ts_, al, be = P.get()
        return torch.cat([X, sensor_solve(X, ts_, be)], 1)

    def loss_fn(parts=False):
        Ld = Lr = L0 = La = 0.0
        for d in data:
            X = states(d); dd = d['drift'](d['t']) if d['drift'] is not None else None
            Ld = Ld + torch.mean((X[:, 2:] - d['y']) ** 2)
            Lr = Lr + torch.mean(heater_residual(X, d['q1'], d['q2'], P, d['TaK'], dd) ** 2)
            L0 = L0 + torch.mean(X[0, :2] ** 2)
            if d['drift'] is not None:
                mg, sm = d['drift'].penalty(); La = La + AMB['w_mag'] * mg + AMB['w_smooth'] * sm
        n = len(data); L = (Ld + w_r * Lr + w_0 * L0 + La) / n
        return (L, Ld / n, Lr / n, L0 / n, La / n) if parts else L

    log, t0 = [], time.time()
    def record(stage, it):
        with torch.no_grad(): L, Ld, Lr, L0, La = loss_fn(True)
        dmax = max([float(d['drift'].values().detach().abs().max()) for d in data if d['drift'] is not None], default=0.0)
        rec = dict(stage=stage, it=it, L=float(L), L_data=float(Ld), L_res=float(Lr), L_0=float(L0), L_amb=float(La),
                   drift_max=dmax, t_s=time.time() - t0, **P.as_dict()); log.append(rec)
        if verbose: print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in rec.items()})

    # 0) arranque de las redes solo con datos
    opt = torch.optim.Adam(nets.parameters(), lr=lr)
    for it in range(max(n_adam // 4, 1)):
        opt.zero_grad(); Ls = 0.0
        for d in data:
            X = states(d); Ls = Ls + torch.mean((X[:, 2:] - d['y'])**2) + torch.mean((X[:, :2] - d['y'])**2)
        Ls.backward(); opt.step()
    # 1) Adam conjunto (w, theta, d)
    groups = [{'params': nets.parameters(), 'lr': lr}, {'params': P.parameters(), 'lr': 1e-2}]
    if len(drifts): groups.append({'params': drifts.parameters(), 'lr': 1e-2})
    opt = torch.optim.Adam(groups)
    sched = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.1 ** (1.0 / max(n_adam, 1)))
    for it in range(n_adam):
        opt.zero_grad(); L = loss_fn(); L.backward(); opt.step(); sched.step()
        if it % 500 == 0: record('adam', it)
    record('adam', n_adam)
    # 2) L-BFGS
    if n_lbfgs > 0:
        opt = torch.optim.LBFGS(list(nets.parameters()) + list(P.parameters()) + list(drifts.parameters()), lr=1.0,
                                max_iter=n_lbfgs, history_size=50, line_search_fn='strong_wolfe',
                                tolerance_grad=1e-10, tolerance_change=1e-13)
        def closure():
            opt.zero_grad(); L = loss_fn(); L.backward(); return L
        opt.step(closure); record('lbfgs', n_lbfgs)
    with torch.no_grad():
        dhat = {d['name']: (d['drift'](d['t']).numpy().copy() if d['drift'] is not None else None) for d in data}
    return P.as_dict(), dhat, log, time.time() - t0

## 4. Modelos de referencia

- **Físico entero / fraccionario (OE-LS):** el mismo modelo 2×2 ajustado por error de salida con ambiente constante (como NB01).
- **FOPDT, SOPDT, FFOPDT** sobre G11 con el experimento de entrenamiento que excita Q1 (C si está, si no A).
- **IC 95 %** de ϑ̂ por linealización del error de salida sobre los datos de entrenamiento (para la fPINN, con su d(t)).

In [ ]:
def oe_residuals(th, exps, drifts=None, L=None):
    out = []
    for e in exps:
        d = None if drifts is None else drifts.get(e['name'])
        T1, T2, _ = sim_frac(e['Q1'], e['Q2'], th, amb0(e), d=d, L=L)
        out += [T1 - e['T1'], T2 - e['T2']]
    return np.concatenate(out)

def ci_linear(th, exps, names, drifts=None, L=None, rel=1e-4):
    r0 = oe_residuals(th, exps, drifts, L); J = []
    for k in names:
        d = dict(th); dh = rel * max(abs(th[k]), 1e-3); d[k] = th[k] + dh
        J.append((oe_residuals(d, exps, drifts, L) - r0) / dh)
    J = np.array(J).T; s2 = r0 @ r0 / max(len(r0) - len(names), 1)
    return dict(zip(names, np.sqrt(np.diag(s2 * np.linalg.pinv(J.T @ J)))))

def fit_phys_oe(exps, frac):
    names = TH_NAMES if frac else POS_NAMES
    lb = [0.1, 0.5, 1e-3, 1e-3, 1.0] + ([0.3, 0.3] if frac else [])
    ub = [50, 200, 0.05, 0.05, 300] + ([1.2, 1.2] if frac else [])
    x0 = np.clip([TH0[k] if k in POS_NAMES else 0.95 for k in names], np.array(lb) * 1.001, np.array(ub) * 0.999)
    mk = lambda x: {**dict(zip(names, x)), **({} if frac else dict(alpha=1.0, beta=1.0))}
    sol = least_squares(lambda x: oe_residuals(mk(x), exps), x0, bounds=(lb, ub), x_scale='jac', max_nfev=OE_NFEV)
    th = mk(sol.x); return th, ci_linear(th, exps, names)

def delayed(u, theta):
    d = theta / TS; n = int(np.floor(d)); f = d - n; N = len(u); u0 = np.zeros(N); u1 = np.zeros(N)
    if n < N: u0[n:] = u[:N - n]
    if n + 1 < N: u1[n + 1:] = u[:N - n - 1]
    return (1 - f) * u0 + f * u1

def sim_fopdt(p, u):
    K, tau, th = p; ud = delayed(u, th); a = np.exp(-TS / tau); y = np.zeros(len(u))
    for k in range(1, len(u)): y[k] = a * y[k - 1] + K * (1 - a) * ud[k - 1]
    return y

def sim_sopdt(p, u):
    K, t1, t2, th = p; ud = delayed(u, th); a1, a2 = np.exp(-TS / t1), np.exp(-TS / t2)
    x = np.zeros(len(u)); y = np.zeros(len(u))
    for k in range(1, len(u)):
        x[k] = a1 * x[k - 1] + K * (1 - a1) * ud[k - 1]; y[k] = a2 * y[k - 1] + (1 - a2) * x[k - 1]
    return y

def sim_ffopdt(p, u):
    K, tau_a, al, th = p; ud = delayed(u, th); N = len(u)
    g = gl_w(al, N); c = tau_a * TS ** (-al); y = np.zeros(N)
    for k in range(1, N): y[k] = (K * ud[k - 1] - c * (g[1:k + 1] @ y[k - 1::-1])) / (c + 1.0)
    return y

SISO = {'FOPDT':  (sim_fopdt,  ['K', 'tau', 'theta'],            [1e-3, 1, 0],         [5, 3000, 300],        [0.77, 175, 25]),
        'SOPDT':  (sim_sopdt,  ['K', 'tau1', 'tau2', 'theta'],   [1e-3, 1, 0.5, 0],    [5, 3000, 1000, 300],  [0.77, 170, 20, 10]),
        'FFOPDT': (sim_ffopdt, ['K', 'tau_a', 'alpha_f', 'theta'], [1e-3, 0.5, 0.3, 0],  [5, 1e5, 1.9, 300],    [0.77, 175, 1.0, 25])}

def fit_siso(name, u, y, n_starts=SISO_STARTS, seed=0):
    fn, names, lb, ub, x0 = SISO[name]; rng = np.random.default_rng(seed); best = None
    for s in range(n_starts):
        p0 = np.array(x0, float) * (1 if s == 0 else rng.uniform(0.6, 1.5, len(x0)))
        if name == 'FFOPDT' and s > 0: p0[2] = rng.uniform(0.6, 1.2); p0[1] = 175 ** p0[2]
        p0 = np.clip(p0, np.array(lb) + 1e-6, np.array(ub) - 1e-6)
        r = least_squares(lambda p: fn(p, u) - y, p0, bounds=(lb, ub), x_scale='jac', max_nfev=300)
        if best is None or r.cost < best.cost: best = r
    return dict(zip(names, best.x))

## 5. Una configuración: entrenar, validar y criterio de H1

`run_learned(partición, ambiente, semilla)` entrena fPINN y PINN entera; `run_reference(partición)` ajusta OE-LS y SISO
(no dependen de la semilla ni del modelo de ambiente). La validación **simula el modelo identificado** sobre los
experimentos de validación con ambiente constante (T∞ = primeras muestras), que es lo que verá NB03.

**H1 reformulada** (decisión de Luis, 6 nov): la fPINN aprende el orden sin imponer fraccionalidad; se reporta si β̂
(y α̂) es distinto de 1 con el criterio de NB01 (|o − 1| ≥ 0,05, IC 95 % que excluye 1, o < 1) y si eso mejora el RMSE
de validación > 10 % frente a la **PINN entera con los mismos datos y el mismo modelo de ambiente**. Se espera β̂ ≈ 1.

In [ ]:
def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b))**2)))

def val_2x2(th, val_names):
    out, allv = {}, []
    for nm in val_names:
        e = EXPS[nm]; T1, T2, _ = sim_frac(e['Q1'], e['Q2'], th, amb0(e), L=L_SUR)
        out[f'RMSE_{nm}_T1'] = rmse(T1, e['T1']); out[f'RMSE_{nm}_T2'] = rmse(T2, e['T2'])
        allv += [out[f'RMSE_{nm}_T1'], out[f'RMSE_{nm}_T2']]
    if not val_names: return {}          # partición final ABC: sin validación externa
    out['RMSE_val_T1'] = float(np.mean([out[f'RMSE_{nm}_T1'] for nm in val_names]))
    out['RMSE_val_mean'] = float(np.mean(allv)); return out

def train_fit_2x2(th, exps, drifts=None):
    r = oe_residuals(th, exps, drifts); return {'RMSE_train': float(np.sqrt(np.mean(r**2)))}

def q1_exp(names):
    # experimento que excita Q1 (C antes que A)
    for nm in ['C', 'A']:
        if nm in names: return nm
    return None

def val_siso(name, par, val_names):
    nm = q1_exp(val_names); fn, names, *_ = SISO[name]
    if nm is None: return {}
    e = EXPS[nm]; y = e['T1'] - e['T1'][:10].mean()
    r = rmse(fn([par[k] for k in names], e['Q1']), y)
    return {f'RMSE_{nm}_T1': r, 'RMSE_val_T1': r}

def order_flag(v, se):
    return bool(abs(v - 1) >= ORDER_TOL and abs(v - 1) > 1.96 * se and v < 1)

def train_exps(split):
    return [cut(EXPS[nm], D_TRAIN_MIN) for nm in SPLITS[split][0]]

def run_reference(split):
    tr, va = SPLITS[split]; exps = train_exps(split); base = dict(split=split, amb='const', seed=-1)
    rows, sur = [], {}
    for frac, label in [(False, 'Phys-int (OE-LS)'), (True, 'Phys-frac (OE-LS)')]:
        t0 = time.time(); th, se = fit_phys_oe(exps, frac)
        rows.append(dict(base, model=label, time_s=time.time() - t0, **th, **{f'se_{k}': v for k, v in se.items()},
                         **train_fit_2x2(th, exps), **val_2x2(th, va)))
        sur['PhysInt_OE' if not frac else 'PhysFrac_OE'] = th
    nm = q1_exp(tr)
    if nm is not None:
        e = cut(EXPS[nm], D_TRAIN_MIN); u = e['Q1']; y = e['T1'] - e['T1'][:10].mean()
        for name in SISO:
            t0 = time.time(); par = fit_siso(name, u, y)
            rows.append(dict(base, model=name, time_s=time.time() - t0, siso_train=nm, **par,
                             RMSE_train=rmse(SISO[name][0]([par[k] for k in SISO[name][1]], u), y), **val_siso(name, par, va)))
            sur[name] = par
    return rows, sur

def run_learned(split, amb, seed):
    tr, va = SPLITS[split]; exps = train_exps(split); base = dict(split=split, amb=amb, seed=seed)
    rows, sur, extra = [], {}, {}
    variants = [(True, 'fPINN')] + ([(False, 'Integer PINN')] if RUN_INT_PINN else [])
    for learn, label in variants:
        th, dhat, log, tt = train_pinn(exps, learn_orders=learn, amb=amb, seed=seed, **PINN)
        names = TH_NAMES if learn else POS_NAMES
        dd = {k: v for k, v in dhat.items() if v is not None} or None
        se = ci_linear(th, exps, names, dd, L_SUR)
        dmax = max([float(np.abs(v).max()) for v in dd.values()], default=0.0) if dd else 0.0
        rows.append(dict(base, model=label, time_s=tt, **th, **{f'se_{k}': v for k, v in se.items()},
                         L_data=log[-1]['L_data'], L_res=log[-1]['L_res'], drift_max=dmax,
                         **train_fit_2x2(th, exps, dd), **val_2x2(th, va)))
        sur['fPINN' if learn else 'IntegerPINN'] = th
        extra[label] = dict(drift=dhat, log=log)
    p = rows[0]
    fa, fb = order_flag(p['alpha'], p['se_alpha']), order_flag(p['beta'], p['se_beta'])
    if RUN_INT_PINN and va:
        i = rows[1]; gain = 100 * (i['RMSE_val_mean'] - p['RMSE_val_mean']) / i['RMSE_val_mean']
    else:
        gain = float('nan')
    p.update(gain_vs_intPINN_pct=gain, frac_alpha=fa, frac_beta=fb,
             verdict=('EVIDENCIA' if (fa or fb) and gain > GAIN_TOL else
                      ('débil' if (fa or fb or gain > GAIN_TOL) else 'sin evidencia')))
    return rows, sur, extra

## 6. Corrida (se guarda y retoma)

Orden: referencias de cada partición, luego fPINN + PINN entera empezando por `AB-C` / `const`. Cada configuración
se agrega a `resultados_kit*.csv` en cuanto termina; si se interrumpe, al volver a correr la celda salta lo ya hecho.
Los sustitutos para NB03 se guardan en `NB02_out/kit/surrogates/` (un JSON por partición, ambiente y semilla) y la
deriva aprendida y el historial de entrenamiento en `drift_*.csv` y `historial_*.csv`.

In [ ]:
def key_order():
    out = []
    for split in sorted(SPLITS, key=lambda s: s != MAIN_SPLIT):
        for amb in AMB_MODELS:
            for seed in SEEDS: out.append((split, amb, seed))
    return out

old = pd.read_csv(RES_CSV) if os.path.exists(RES_CSV) else pd.DataFrame(columns=['split', 'amb', 'seed', 'model'])
done_ref = set(old[old.seed == -1].split) if len(old) else set()
done_l = set(map(tuple, old[old.model == 'fPINN'][['split', 'amb', 'seed']].values.tolist())) if len(old) else set()

def append(rows):
    # reescribe el CSV completo: las filas de referencia y las de la fPINN tienen columnas distintas
    df = pd.DataFrame(rows)
    allr = pd.concat([pd.read_csv(RES_CSV), df], ignore_index=True) if os.path.exists(RES_CSV) else df
    allr.to_csv(RES_CSV, index=False); return df

REF_SUR = {}
for split in sorted(SPLITS, key=lambda s: s != MAIN_SPLIT):
    jf = os.path.join(OUT_DIR, 'surrogates', f'ref_{split}.json')
    if split in done_ref and os.path.exists(jf):
        REF_SUR[split] = json.load(open(jf))['models']; continue
    t0 = time.time(); rows, sur = run_reference(split); df = append(rows); REF_SUR[split] = sur
    json.dump(dict(split=split, train=SPLITS[split][0], phys=PHYS, h=TS, models=sur), open(jf, 'w'), indent=1)
    print(f'[ref {split}] ' + '  '.join(f"{r.model}: {'val T1' if SPLITS[split][1] else 'ajuste'} {getattr(r, 'RMSE_val_T1', getattr(r, 'RMSE_train', float('nan'))):.3f}" for r in df.itertuples()) +
          f'  ({time.time() - t0:.0f} s)')

todo = [k for k in key_order() if k not in done_l]
print(f'{len(todo)} configuraciones de fPINN pendientes de {len(key_order())}')
t_start = time.time()
for i, (split, amb, seed) in enumerate(todo):
    t0 = time.time()
    rows, sur, extra = run_learned(split, amb, seed); df = append(rows)
    tag = f'{split}_{amb}_s{seed}'
    json.dump(dict(split=split, train=SPLITS[split][0], val=SPLITS[split][1], amb=amb, seed=seed, D_train_min=D_TRAIN_MIN,
                   phys=PHYS, h=TS, L=L_SUR, T_AMB_NB01=META.get('T_AMB'),
                   note='theta identificado; la deriva d(t) es solo del entrenamiento y no forma parte del sustituto',
                   models={**sur, **REF_SUR.get(split, {})}),
              open(os.path.join(OUT_DIR, 'surrogates', f'{tag}.json'), 'w'), indent=1)
    for label, ex in extra.items():
        lab = label.replace(' ', '')
        pd.DataFrame(ex['log']).to_csv(os.path.join(OUT_DIR, f'historial_{lab}_{tag}.csv'), index=False)
        dr = {nm: v for nm, v in ex['drift'].items() if v is not None}
        if dr:
            n = max(len(v) for v in dr.values())
            pd.DataFrame({f'd_{nm}': np.r_[v, np.full(n - len(v), np.nan)] for nm, v in dr.items()}).assign(
                t=np.arange(n) * TS).to_csv(os.path.join(OUT_DIR, f'drift_{lab}_{tag}.csv'), index=False)
    p = df[df.model == 'fPINN'].iloc[0]
    q = df[df.model == 'Integer PINN'].iloc[0] if RUN_INT_PINN else None
    print(f'[{i + 1}/{len(todo)}] {tag}: α̂={p.alpha:.3f}±{1.96 * p.se_alpha:.3f} β̂={p.beta:.3f}±{1.96 * p.se_beta:.3f} '
          f'U={p.U:.2f} Us={p.Us:.1f} τs={p.tau_s:.1f} | {"val" if SPLITS[split][1] else "ajuste"} fPINN {p.get("RMSE_val_mean", p.RMSE_train):.3f}'
          + (f', PINN entera {q.get("RMSE_val_mean", q.RMSE_train):.3f}' if q is not None else '') +
          f' °C | {p.verdict} ({time.time() - t0:.0f} s; total {(time.time() - t_start) / 60:.1f} min)')
res = pd.read_csv(RES_CSV)
for c in ['alpha', 'beta', 'RMSE_val_T1', 'RMSE_val_mean', 'RMSE_train', 'se_beta', 'gain_vs_intPINN_pct', 'verdict']:
    if c not in res: res[c] = np.nan

## 7. Tabla de identificación con datos reales (media ± desv. estándar sobre semillas)

Una fila por partición, modelo de ambiente y modelo. Se guarda como CSV y como filas LaTeX para el manuscrito
(tabla de identificación experimental; el hilo del manuscrito las incorpora).

In [ ]:
pm = lambda s, d=3: (f'{s.mean():.{d}f} \\pm {s.std(ddof=1):.{d}f}' if s.notna().sum() > 1 else
                     (f'{s.mean():.{d}f}' if s.notna().any() else 'n/a'))
order = ['fPINN', 'Integer PINN', 'Phys-frac (OE-LS)', 'Phys-int (OE-LS)', 'FFOPDT', 'SOPDT', 'FOPDT']
tab, tex = [], []
for split in SPLITS:
    for amb in sorted(res[res.split == split].amb.unique()):
        for model in order:
            g = res[(res.split == split) & (res.amb == amb) & (res.model == model)]
            if not len(g): continue
            row = dict(split=split, amb=amb, model=model, n=len(g),
                       alpha=pm(g.alpha) if 'alpha' in g else 'n/a', beta=pm(g.beta) if 'beta' in g else 'n/a',
                       RMSE_val_T1=pm(g.RMSE_val_T1),
                       RMSE_val_mean=pm(g.RMSE_val_mean) if 'RMSE_val_mean' in g and g.RMSE_val_mean.notna().any() else 'n/a',
                       RMSE_train=pm(g.RMSE_train) if 'RMSE_train' in g and g.RMSE_train.notna().any() else 'n/a',
                       time_s=f'{g.time_s.mean():.0f}')
            tab.append(row)
            tex.append(f"{split} & {amb} & {model} & ${row['alpha']}$ & ${row['beta']}$ & ${row['RMSE_val_T1']}$ & "
                       f"${row['RMSE_val_mean']}$ \\\\")
tab = pd.DataFrame(tab); tab.to_csv(os.path.join(OUT_DIR, 'tab_kit_id.csv'), index=False)
open(os.path.join(OUT_DIR, 'tab_kit_id_rows.tex'), 'w').write('\n'.join(tex) + '\n')
print(tab.to_string(index=False))

f = res[res.model == 'fPINN']
ver = f.groupby(['split', 'amb']).agg(alpha_hat=('alpha', 'mean'), beta_hat=('beta', 'mean'), se_beta=('se_beta', 'mean'),
                                      gain_vs_intPINN_pct=('gain_vs_intPINN_pct', 'mean'),
                                      evidencia=('verdict', lambda v: f"{(v == 'EVIDENCIA').sum()}/{len(v)}"))
ver.to_csv(os.path.join(OUT_DIR, 'veredicto_H1_kit.csv')); print(); print(ver.round(4))

## 8. Figuras: validación, deriva aprendida y entrenamiento

In [ ]:
split = MAIN_SPLIT; amb = AMB_MODELS[0]; seed = SEEDS[0]; tag = f'{split}_{amb}_s{seed}'
S = json.load(open(os.path.join(OUT_DIR, 'surrogates', f'{tag}.json')))['models']
va = SPLITS[split][1] or SPLITS[split][0]     # en ABC (sin validación) se muestra el ajuste a los datos de entrenamiento
fig, ax = plt.subplots(2, len(va), figsize=(7 * len(va), 5), squeeze=False, sharex='col')
for j, nm in enumerate(va):
    e = EXPS[nm]; Ta = amb0(e)
    ax[0, j].plot(e['t'], e['T1'], c='k', lw=1, label='medido'); ax[1, j].plot(e['t'], e['T2'], c='k', lw=1)
    for mname, lab in [('fPINN', 'fPINN'), ('IntegerPINN', 'PINN entera'), ('PhysInt_OE', 'físico entero OE-LS')]:
        if mname not in S: continue
        T1, T2, _ = sim_frac(e['Q1'], e['Q2'], S[mname], Ta, L=L_SUR)
        ax[0, j].plot(e['t'], T1, lw=1, label=f'{lab} ({rmse(T1, e["T1"]):.2f} °C)'); ax[1, j].plot(e['t'], T2, lw=1)
    if 'SOPDT' in S and nm in ('A', 'C'):
        p = S['SOPDT']; y = sim_sopdt([p[k] for k in SISO['SOPDT'][1]], e['Q1']) + e['T1'][:10].mean()
        ax[0, j].plot(e['t'], y, '--', lw=1, label=f'SOPDT G11 ({rmse(y, e["T1"]):.2f} °C)')
    ax[0, j].set_title(f"{'Validación' if SPLITS[split][1] else 'Ajuste (entrenamiento)'}: exp. {nm} ({tag})"); ax[0, j].legend(fontsize=7)
    ax[0, j].set_ylabel('T1 [°C]'); ax[1, j].set_ylabel('T2 [°C]'); ax[1, j].set_xlabel('t [s]')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig02_validacion_kit.png'), dpi=150)
plt.savefig(os.path.join(OUT_DIR, 'fig02_validacion_kit.pdf')); plt.show(); plt.close('all')

dfiles = sorted(glob.glob(os.path.join(OUT_DIR, 'drift_fPINN_*.csv')))
if dfiles:
    fig, ax = plt.subplots(1, 1, figsize=(8, 3))
    for fn in dfiles:
        d = pd.read_csv(fn); tg = os.path.basename(fn)[len('drift_fPINN_'):-4]
        for c in [c for c in d.columns if c.startswith('d_')]: ax.plot(d.t, d[c], lw=1, label=f'{tg} {c[2:]}')
    ax.set_xlabel('t [s]'); ax.set_ylabel('d(t) [°C]'); ax.set_title('Deriva del ambiente aprendida (fPINN)')
    ax.legend(fontsize=6, ncol=2); plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig03_deriva_ambiente.png'), dpi=150); plt.show(); plt.close('all')

h = pd.read_csv(os.path.join(OUT_DIR, f'historial_fPINN_{tag}.csv'))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for c in ['L_data', 'L_res', 'L_0', 'L_amb']: ax[0].semilogy(np.arange(len(h)), h[c].clip(lower=1e-12), marker='.', label=c)
ax[0].set_xlabel('registro (cada 500 it de Adam + L-BFGS)'); ax[0].legend(fontsize=7)
ax[1].plot(h.alpha, marker='.', label='α'); ax[1].plot(h.beta, marker='.', label='β'); ax[1].axhline(1, c='0.6', lw=0.8)
ax[1].legend(); ax[1].set_title('órdenes durante el entrenamiento')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig04_entrenamiento.png'), dpi=150); plt.show(); plt.close('all')

## 9. Empaquetar resultados

Crea `NB02_kit_resultados_<fecha>.zip` con todo `NB02_out/kit/` (CSV, tablas, sustitutos JSON, derivas, historiales,
figuras y un `metadatos_nb02.json`). Ese .zip es el que se sube al proyecto para analizarlo.

In [ ]:
meta2 = dict(fecha=datetime.datetime.now().isoformat(timespec='seconds'), mode=MODE, fuente_datos=str(SRC),
             splits=SPLITS, amb_models=AMB_MODELS, seeds=SEEDS, D_train_min=D_TRAIN_MIN, pinn=PINN, amb=AMB,
             oe_nfev=OE_NFEV, th0=TH0, phys=PHYS, so=platform.platform(), python=platform.python_version(),
             torch=torch.__version__, procesador=platform.processor(), hilos_torch=torch.get_num_threads(),
             kit=dict((k, META.get(k)) for k in ['fecha', 'firmware', 'T_AMB']))
json.dump(meta2, open(os.path.join(OUT_DIR, 'metadatos_nb02.json'), 'w'), indent=1, default=str)
zname = f"NB02_kit_resultados_{datetime.date.today().strftime('%Y%m%d')}{'_ABC' if FINAL_ABC else ''}{'_quick' if MODE == 'quick' else ''}.zip"
with zipfile.ZipFile(zname, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT_DIR):
        for fn in files: z.write(os.path.join(root, fn))
print('Guardado:', os.path.abspath(zname), f'({os.path.getsize(zname) / 1e6:.1f} MB)')

## 10. Notas para la discusión

- **Orden esperado ≈ 1.** NB01 con datos completos dio β = 1,00 ± 0,01 en el modelo físico. Si la fPINN también da
  α̂, β̂ ≈ 1, el resultado se reporta así: el método **aprende** el orden y no inventa fraccionalidad. El aporte del
  artículo queda en la sintonía diferenciable del FOPID (λ, μ del controlador) y en H2/H3.
- **fPINN frente a PINN entera (B3).** Con β̂ ≈ 1 ambas deberían empatar en identificación; la diferencia relevante
  aparece en NB03/NB04 (controlador fraccionario frente a PID sobre el mismo tipo de sustituto).
- **Deriva del ambiente.** Comparar `drift` frente a `const`: si `drift` cambia ϑ̂ (sobre todo U, τs) y mejora la
  validación, conviene usarlo en el texto; si solo baja el error de entrenamiento, la deriva está absorbiendo error de
  modelo (es lo que pasó en la prueba intermedia: d(t) siguió a Q). Revisar `fig03_deriva_ambiente.png`. La solución
  de fondo es medir la sala con un termómetro aparte (o cubrir el kit) en las próximas corridas.
- **Resolución del sensor:** 0,032 °C (firmware 2.0.1), ruido ≈ 0,03 °C. Aquí no se agrega ruido ni cuantización.